# Reranking and evidence selection

**Track:** Enterprise Knowledge Assistant · **Stage:** Retrieval engineering

First-stage retrieval optimizes recall and speed. Reranking optimizes final evidence quality. NovaTech’s query asks whether the Atlas supplier must satisfy Regulation R-17. A broad candidate set may contain project docs, vendor docs, unrelated policies, and exact but incomplete snippets.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart LR
  Q["Question"] --> R1["First-stage retrieve top 30"]
  R1 --> N["Noisy candidates"]
  N --> R2["Rerank / evidence selection"]
  R2 --> K["Small grounded context"]
  K --> A["Answer + citations"]
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

The code below approximates a reranker with transparent evidence-term scoring. In production, you might replace that function with a cross-encoder from Sentence Transformers, Cohere Rerank, BGE reranker, Haystack ranker, or a late-interaction model.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, explain_hits, rerank_by_evidence_terms
from src.enterprise_rag.retrieval import hybrid_retrieve
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
question = "Does the Atlas supplier need R-17 certification?"
candidates = hybrid_retrieve(question, chunks, top_k=30)
print("Before reranking")
show(explain_hits(candidates, limit=8))
print("\nAfter reranking")
reranked = rerank_by_evidence_terms(question, candidates, extra_terms=["Acme", "VectorDB-X", "Regulation", "R-17", "certify"])
show(explain_hits(reranked, limit=8))

### Design rule

Do not rerank the world. Retrieve broadly enough to get recall, rerank narrowly enough to control latency and cost, and evaluate whether the final context contains the exact evidence needed to support the answer.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/